In [1]:
!pip install datasets

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
from dataclasses import dataclass

# ── RMSNorm ─────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_f = x.float()
        norm = x_f * torch.rsqrt(x_f.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.scale).to(x.dtype)

# ── Attention ───────────────────────────────────
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)

        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

# ── MLP ─────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

# ── Block ───────────────────────────────────────
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = RMSNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln2 = RMSNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# ── Config ──────────────────────────────────────
@dataclass
class VidhiConfig:
    block_size: int = 512
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 8
    n_embd: int = 512

# ── Model ───────────────────────────────────────
class Vidhi(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)

        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd)

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()

        pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            return logits, None

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.view(-1)
        )

        return logits, loss

In [5]:
from datasets import load_dataset
import tiktoken
import numpy as np
from tqdm import tqdm

enc = tiktoken.get_encoding("gpt2")

stream = load_dataset("MedRAG/pubmed", split="train", streaming=True)

N = 2_000_000
max_tokens = N * 300

arr = np.memmap("train.bin", dtype=np.uint16, mode="w+", shape=(max_tokens,))

idx = 0
count = 0

for example in tqdm(stream, total=N):
    if count >= N:
        break

    text = (example.get("contents", "") or "").strip()
    if len(text) < 100:
        continue

    ids = enc.encode_ordinary(text)

    if len(ids) < 50:
        continue

    arr[idx:idx+len(ids)] = ids
    idx += len(ids)
    count += 1

arr.flush()
print(f"✅ Tokens: {idx/1e6:.1f}M")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1166 [00:00<?, ?it/s]

2008043it [11:16, 2969.49it/s]                             


✅ Tokens: 492.6M


In [6]:
import os, glob
for f in glob.glob("checkpoints/*.pt"):
    os.remove(f)
print("🗑️ Checkpoints cleared")

🗑️ Checkpoints cleared


In [8]:
import torch
import math
import numpy as np
import os

# ─────────────────────────────────────────────
# 1. Setup
# ─────────────────────────────────────────────
device = "cuda"

block_size = 512
batch_size = 16        # for 2 GPUs
grad_accum = 8
max_iters = 50000

lr = 3e-4
min_lr = 3e-5
warmup = 2000

# ─────────────────────────────────────────────
# 2. Model
# ─────────────────────────────────────────────
config = VidhiConfig()

raw_model = Vidhi(config).to(device)

if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(raw_model)
else:
    model = raw_model

optimizer = torch.optim.AdamW(raw_model.parameters(), lr=lr, weight_decay=0.1)

# ─────────────────────────────────────────────
# 3. Data
# ─────────────────────────────────────────────
data = np.memmap("train.bin", dtype=np.uint16, mode="r")

def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([
        torch.from_numpy(data[i:i+block_size]).long()
        for i in ix
    ]).to(device)

    y = torch.stack([
        torch.from_numpy(data[i+1:i+1+block_size]).long()
        for i in ix
    ]).to(device)

    return x, y

# ─────────────────────────────────────────────
# 4. LR Schedule
# ─────────────────────────────────────────────
def get_lr(it):
    if it < warmup:
        return lr * it / warmup
    decay = (it - warmup) / (max_iters - warmup)
    return min_lr + 0.5 * (1 + math.cos(math.pi * decay)) * (lr - min_lr)

# ─────────────────────────────────────────────
# 5. Evaluation
# ─────────────────────────────────────────────
@torch.no_grad()
def estimate_loss(eval_iters=100):
    model.eval()
    losses = []

    for _ in range(eval_iters):
        x, y = get_batch()
        _, loss = model(x, y)
        loss = loss.mean()   # 🔥 IMPORTANT for multi-GPU
        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)

# ─────────────────────────────────────────────
# 6. Checkpointing
# ─────────────────────────────────────────────
best_val_loss = float("inf")
best_ckpt_path = None
os.makedirs("checkpoints", exist_ok=True)

# ─────────────────────────────────────────────
# 7. Training Loop
# ─────────────────────────────────────────────
print("\n🚀 Starting Vidhi training...")

for it in range(max_iters):

    # update LR
    lr_now = get_lr(it)
    for g in optimizer.param_groups:
        g["lr"] = lr_now

    # forward
    x, y = get_batch()
    logits, loss = model(x, y)

    # 🔥 FIX for multi-GPU
    loss = loss.mean()

    # gradient accumulation
    loss = loss / grad_accum
    loss.backward()

    # optimizer step
    if (it + 1) % grad_accum == 0:
        torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    # ── Eval + Save ─────────────────────────
    if it % 500 == 0 and it != 0:

        val_loss = estimate_loss()
        train_loss = loss.item() * grad_accum

        print(f"[iter {it}] train {train_loss:.4f} | val {val_loss:.4f} | lr {lr_now:.2e}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            new_path = f"checkpoints/vidhi_best_{it}.pt"
            torch.save(raw_model.state_dict(), new_path)

            print(f"💾 Saved BEST → {new_path}")

            if best_ckpt_path is not None and os.path.exists(best_ckpt_path):
                os.remove(best_ckpt_path)
                print(f"🗑️ Deleted old → {best_ckpt_path}")

            best_ckpt_path = new_path

print("\n✅ Training complete!")


🚀 Using 2 GPUs

🚀 Starting Vidhi training...


KeyboardInterrupt: 

In [9]:
import os

ckpt_dir = "checkpoints"

files = os.listdir(ckpt_dir)

print("📦 Files in checkpoints:")
for f in files:
    print(f)

📦 Files in checkpoints:
vidhi_best_32500.pt


In [10]:
import torch
import tiktoken

# ─────────────────────────────────────────────
# 1. Setup
# ─────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

enc = tiktoken.get_encoding("gpt2")

def encode(text):
    return enc.encode_ordinary(text)

def decode(tokens):
    return enc.decode(tokens)

# ─────────────────────────────────────────────
# 2. Load Model
# ─────────────────────────────────────────────
config = VidhiConfig()
model = Vidhi(config)

ckpt_path = "checkpoints/vidhi_best_32500.pt"

state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)

model.to(device)
model.eval()

print("✅ Model loaded from best checkpoint")

# ─────────────────────────────────────────────
# 3. Generation Function
# ─────────────────────────────────────────────
@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=150,
    temperature=0.7,
    top_k=40,
    top_p=0.9
):
    input_ids = torch.tensor([encode(prompt)], dtype=torch.long).to(device)

    for _ in range(max_new_tokens):
        idx_cond = input_ids[:, -512:]

        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        # top-k filtering
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        # top-p (nucleus) filtering
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
            sorted_indices_to_remove[:, 0] = False

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[:, indices_to_remove] = -float("inf")

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    return decode(input_ids[0].tolist())

# ─────────────────────────────────────────────
# 4. Test Prompts
# ─────────────────────────────────────────────

def run_test(prompt):
    print("\n📝 Prompt:", prompt)
    print("─" * 60)
    output = generate(prompt)
    print(output)
    print("─" * 60)

# Medical prompts
run_test("Dengue fever is a mosquito-borne viral infection characterized by")

run_test("The mechanism of action of Metformin in Type 2 Diabetes involves")

run_test("A 45 year old patient presents with chest pain and shortness of breath.")

✅ Model loaded from best checkpoint

📝 Prompt: Dengue fever is a mosquito-borne viral infection characterized by
────────────────────────────────────────────────────────────
Dengue fever is a mosquito-borne viral infection characterized by an immunological response against the parasite. In this study, we developed a virus-like virus (VV)-like particles and evaluated the virus-neutralizing (VV) and the viral-antibody (VV) responses to the VV genome. In a random sample of a mosquito-derived mosquito-derived VV-like particles, a VV-like particle was found to be present in the plasma membrane of the VV-like particles, but not in the plasma membrane of the VV-like particles. The VV-like particle was found to be present in the plasma membrane of the VV-like particles in the plasma membrane of the VV-like particles. The virus-like particles were
────────────────────────────────────────────────────────────

📝 Prompt: The mechanism of action of Metformin in Type 2 Diabetes involves
────────────